## Lab: Building and Querying a Relational Healthcare Database Using SQL

**Estimated Time:** 20 minutes

## Introduction

Healthcare organizations rely on relational databases to store and connect information about patients, visits, diagnoses, billing, and lab results.

Instead of keeping all data in a single large table, relational databases organize information into smaller, linked tables that:

- Reduce errors and duplication  
- Improve consistency  
- Make querying much more efficient  

In this lab, you will:

- Build normalized tables using SQL
- Insert sample patient and visit data
- Run queries to answer operational questions

## Learning Objectives

By the end of this lab, you will be able to:

- Organize healthcare data into normalized relational tables  
- Define primary and foreign keys to link patients, visits/billing, and lab results  
- Use SQL to create and modify tables  
- Query the database using `SELECT`, `WHERE`, `ORDER BY`, and `JOIN`

## Setup: Create a SQLite Database in Python

This notebook uses **SQLite** via Python's built-in `sqlite3` module.

Run the cell below to:

- Import required libraries  
- Create a SQLite database file (`healthcare_lab.db`)  
- Create helper functions to execute and display SQL queries

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

# Create / connect to a local SQLite database file
db_path = Path("healthcare_lab.db")
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

print(f"Connected to SQLite database at: {db_path.resolve()}")

def run_sql(sql: str):
    """Execute one or more SQL statements (no result returned)."""
    cursor.executescript(sql)
    conn.commit()

def show_query(sql: str):
    """Run a SELECT query and display results as a pandas DataFrame."""
    print("SQL query:")
    print(sql.strip())
    df = pd.read_sql_query(sql, conn)
    display(df)
    return df

Connected to SQLite database at: /content/healthcare_lab.db


## Task 1: Create the Healthcare Database Tables

We will create three tables:

1. `Patients` – one row per patient  
2. `Visits` – one row per visit (billing record)  
3. `LabTestResults` – one row per lab result, linked to both patient and visit

### 1.1 Create the `Patients` Table (SQLite)

In [2]:
patients_sql = '''
DROP TABLE IF EXISTS Patients;

CREATE TABLE Patients (
    PatientID TEXT PRIMARY KEY,
    Name TEXT,
    DOB TEXT NOT NULL,
    Gender TEXT,
    Address TEXT
);
'''

run_sql(patients_sql)
print("Patients table created.")

Patients table created.


### 1.2 Create the `Visits` Table (SQLite)

In [3]:
visits_sql = '''
DROP TABLE IF EXISTS Visits;

CREATE TABLE Visits (
    VisitID TEXT PRIMARY KEY,
    PatientID TEXT NOT NULL,
    VisitDate TEXT,
    ProcedureCode TEXT,
    ProcedureDescription TEXT,
    Charge REAL,
    Payer TEXT,
    FOREIGN KEY (PatientID) REFERENCES Patients(PatientID)
);
'''

run_sql(visits_sql)
print("Visits table created.")

Visits table created.


### 1.3 Create the `LabTestResults` Table (SQLite)

In [4]:
lab_sql = '''
DROP TABLE IF EXISTS LabTestResults;

CREATE TABLE LabTestResults (
    LabResultID TEXT PRIMARY KEY,
    VisitID TEXT NOT NULL,
    PatientID TEXT NOT NULL,
    LabTest TEXT,
    ResultValue TEXT,
    ResultDate TEXT,
    FOREIGN KEY (VisitID) REFERENCES Visits(VisitID),
    FOREIGN KEY (PatientID) REFERENCES Patients(PatientID)
);
'''

run_sql(lab_sql)
print("LabTestResults table created.")

LabTestResults table created.


## Task 2: Insert Data into Tables

In this task, you will add the available data into the three tables.

### 2.1 Insert Data into `Patients`

In [5]:
insert_patients_sql = '''
INSERT INTO Patients (PatientID, Name, DOB, Gender, Address) VALUES
('P001', 'John Smith', '1985-02-14', 'Male', '1245 Willow Creek Drive, Raleigh, NC 27603'),
('P002', 'Maria Lopez', '1990-11-03', 'Female', '892 North Ridge Avenue, Durham, NC 27701'),
('P003', 'Tim Chen', '1978-06-22', 'Male', '2108 Maplewood Lane, Chapel Hill, NC 27514'),
('P004', 'Aisha Khan', '1990-09-09', 'Female', '55 Harbor Point Road, Charlotte, NC 28202');
'''

run_sql(insert_patients_sql)
print("Sample patients inserted.")

# Verify
show_query("SELECT * FROM Patients;")

Sample patients inserted.
SQL query:
SELECT * FROM Patients;


,PatientID,Name,DOB,Gender,Address
0,P001,John Smith,1985-02-14,Male,"1245 Willow Creek Drive, Raleigh, NC 27603"
1,P002,Maria Lopez,1990-11-03,Female,"892 North Ridge Avenue, Durham, NC 27701"
2,P003,Tim Chen,1978-06-22,Male,"2108 Maplewood Lane, Chapel Hill, NC 27514"
3,P004,Aisha Khan,1990-09-09,Female,"55 Harbor Point Road, Charlotte, NC 28202"


,PatientID,Name,DOB,Gender,Address
0,P001,John Smith,1985-02-14,Male,"1245 Willow Creek Drive, Raleigh, NC 27603"
1,P002,Maria Lopez,1990-11-03,Female,"892 North Ridge Avenue, Durham, NC 27701"
2,P003,Tim Chen,1978-06-22,Male,"2108 Maplewood Lane, Chapel Hill, NC 27514"
3,P004,Aisha Khan,1990-09-09,Female,"55 Harbor Point Road, Charlotte, NC 28202"


### 2.2 Insert Data into `Visits` (Billing)

In [6]:
insert_visits_sql = '''
INSERT INTO Visits (VisitID, PatientID, ProcedureCode, ProcedureDescription, Charge, Payer, VisitDate) VALUES
('V001', 'P001', '85025', 'Complete Blood Count (CBC)', 45.00, 'Aetna', '2024-01-12'),
('V002', 'P002', '99213', 'Office Visit – Established Patient', 90.00, 'BlueCross', '2024-01-15'),
('V003', 'P003', '36415', 'Venipuncture', 15.00, 'Medicare', '2024-01-20'),
('V004', 'P001', '80053', 'Comprehensive Metabolic Panel (CMP)', 55.00, 'Aetna', '2024-02-01'),
('V005', 'P004', '87880', 'Rapid Strep Test', 30.00, 'Cigna', '2024-02-05');
'''

run_sql(insert_visits_sql)
print("Sample visits inserted.")

# Verify
show_query("SELECT * FROM Visits;")

Sample visits inserted.
SQL query:
SELECT * FROM Visits;


,VisitID,PatientID,VisitDate,ProcedureCode,ProcedureDescription,Charge,Payer
0,V001,P001,2024-01-12,85025,Complete Blood Count (CBC),45.0,Aetna
1,V002,P002,2024-01-15,99213,Office Visit – Established Patient,90.0,BlueCross
2,V003,P003,2024-01-20,36415,Venipuncture,15.0,Medicare
3,V004,P001,2024-02-01,80053,Comprehensive Metabolic Panel (CMP),55.0,Aetna
4,V005,P004,2024-02-05,87880,Rapid Strep Test,30.0,Cigna


,VisitID,PatientID,VisitDate,ProcedureCode,ProcedureDescription,Charge,Payer
0,V001,P001,2024-01-12,85025,Complete Blood Count (CBC),45.0,Aetna
1,V002,P002,2024-01-15,99213,Office Visit – Established Patient,90.0,BlueCross
2,V003,P003,2024-01-20,36415,Venipuncture,15.0,Medicare
3,V004,P001,2024-02-01,80053,Comprehensive Metabolic Panel (CMP),55.0,Aetna
4,V005,P004,2024-02-05,87880,Rapid Strep Test,30.0,Cigna


### 2.3 Insert Data into `LabTestResults`

In [7]:
insert_labs_sql = '''
INSERT INTO LabTestResults (LabResultID, VisitID, PatientID, LabTest, ResultValue, ResultDate) VALUES
('LR001', 'V001', 'P001', 'CBC – Hemoglobin', '13.5 g/dL', '2024-01-12'),
('LR002', 'V001', 'P001', 'CBC – WBC Count', '7.8 x10^3/uL', '2024-01-12'),
('LR003', 'V003', 'P003', 'Venipuncture – Sample Drawn', 'Completed', '2024-01-20'),
('LR004', 'V004', 'P001', 'CMP – Glucose', '92 mg/dL', '2024-02-01'),
('LR005', 'V005', 'P004', 'Rapid Strep Test', 'Negative', '2024-02-05');
'''

run_sql(insert_labs_sql)
print("Sample lab test results inserted.")

# Verify
show_query("SELECT * FROM LabTestResults;")

Sample lab test results inserted.
SQL query:
SELECT * FROM LabTestResults;


,LabResultID,VisitID,PatientID,LabTest,ResultValue,ResultDate
0,LR001,V001,P001,CBC – Hemoglobin,13.5 g/dL,2024-01-12
1,LR002,V001,P001,CBC – WBC Count,7.8 x10^3/uL,2024-01-12
2,LR003,V003,P003,Venipuncture – Sample Drawn,Completed,2024-01-20
3,LR004,V004,P001,CMP – Glucose,92 mg/dL,2024-02-01
4,LR005,V005,P004,Rapid Strep Test,Negative,2024-02-05


,LabResultID,VisitID,PatientID,LabTest,ResultValue,ResultDate
0,LR001,V001,P001,CBC – Hemoglobin,13.5 g/dL,2024-01-12
1,LR002,V001,P001,CBC – WBC Count,7.8 x10^3/uL,2024-01-12
2,LR003,V003,P003,Venipuncture – Sample Drawn,Completed,2024-01-20
3,LR004,V004,P001,CMP – Glucose,92 mg/dL,2024-02-01
4,LR005,V005,P004,Rapid Strep Test,Negative,2024-02-05


## Task 3: Query the Database

Now that the data is loaded, we'll run a series of queries to answer typical healthcare analytics questions.

### Query 1: Which patients had Complete Blood Count (CBC) procedures?

We want to find patients whose billing information includes the term  
`'Complete Blood Count (CBC)'` in the `ProcedureDescription` field.

In [8]:
query1 = '''
SELECT Name, DOB
FROM Patients
WHERE PatientID IN (
    SELECT PatientID
    FROM Visits
    WHERE ProcedureDescription LIKE '%Complete Blood Count (CBC)%'
);
'''

show_query(query1)

SQL query:
SELECT Name, DOB
FROM Patients
WHERE PatientID IN (
    SELECT PatientID
    FROM Visits
    WHERE ProcedureDescription LIKE '%Complete Blood Count (CBC)%'
);


,Name,DOB
0,John Smith,1985-02-14


,Name,DOB
0,John Smith,1985-02-14


### Query 2: Sort all patients alphabetically by name

In [9]:
query2 = '''
SELECT Name, DOB, Gender
FROM Patients
ORDER BY Name;
'''

show_query(query2)

SQL query:
SELECT Name, DOB, Gender
FROM Patients
ORDER BY Name;


,Name,DOB,Gender
0,Aisha Khan,1990-09-09,Female
1,John Smith,1985-02-14,Male
2,Maria Lopez,1990-11-03,Female
3,Tim Chen,1978-06-22,Male


,Name,DOB,Gender
0,Aisha Khan,1990-09-09,Female
1,John Smith,1985-02-14,Male
2,Maria Lopez,1990-11-03,Female
3,Tim Chen,1978-06-22,Male


### Query 3: Show each patient with their visit dates (LEFT JOIN)

Use a `LEFT JOIN` so that:

- All patients appear  
- Even those who do not have visits yet  

Sort by `Name` and `VisitDate` for readability.

In [10]:
query3 = '''
SELECT
    Patients.Name,
    Visits.VisitDate
FROM Patients
LEFT JOIN Visits
    ON Patients.PatientID = Visits.PatientID
ORDER BY Patients.Name, Visits.VisitDate;
'''

show_query(query3)

SQL query:
SELECT
    Patients.Name,
    Visits.VisitDate
FROM Patients
LEFT JOIN Visits
    ON Patients.PatientID = Visits.PatientID
ORDER BY Patients.Name, Visits.VisitDate;


,Name,VisitDate
0,Aisha Khan,2024-02-05
1,John Smith,2024-01-12
2,John Smith,2024-02-01
3,Maria Lopez,2024-01-15
4,Tim Chen,2024-01-20


,Name,VisitDate
0,Aisha Khan,2024-02-05
1,John Smith,2024-01-12
2,John Smith,2024-02-01
3,Maria Lopez,2024-01-15
4,Tim Chen,2024-01-20


## (Optional) Inspect Table Schemas

Use SQLite PRAGMA commands to inspect the structure of each table.

In [11]:
show_query("PRAGMA table_info(Patients);")
show_query("PRAGMA table_info(Visits);")
show_query("PRAGMA table_info(LabTestResults);")

SQL query:
PRAGMA table_info(Patients);


,cid,name,type,notnull,dflt_value,pk
0,0,PatientID,TEXT,0,None,1
1,1,Name,TEXT,0,None,0
2,2,DOB,TEXT,1,None,0
3,3,Gender,TEXT,0,None,0
4,4,Address,TEXT,0,None,0


SQL query:
PRAGMA table_info(Visits);


,cid,name,type,notnull,dflt_value,pk
0,0,VisitID,TEXT,0,None,1
1,1,PatientID,TEXT,1,None,0
2,2,VisitDate,TEXT,0,None,0
3,3,ProcedureCode,TEXT,0,None,0
4,4,ProcedureDescription,TEXT,0,None,0
5,5,Charge,REAL,0,None,0
6,6,Payer,TEXT,0,None,0


SQL query:
PRAGMA table_info(LabTestResults);


,cid,name,type,notnull,dflt_value,pk
0,0,LabResultID,TEXT,0,None,1
1,1,VisitID,TEXT,1,None,0
2,2,PatientID,TEXT,1,None,0
3,3,LabTest,TEXT,0,None,0
4,4,ResultValue,TEXT,0,None,0
5,5,ResultDate,TEXT,0,None,0


,cid,name,type,notnull,dflt_value,pk
0,0,LabResultID,TEXT,0,None,1
1,1,VisitID,TEXT,1,None,0
2,2,PatientID,TEXT,1,None,0
3,3,LabTest,TEXT,0,None,0
4,4,ResultValue,TEXT,0,None,0
5,5,ResultDate,TEXT,0,None,0


# Exercises

Now it's your turn! Apply what you've learned to the healthcare dataset. The following exercises will test your understanding of the SQL queries.

## Exercise 1
Write an SQL query to retrieve all female patients from the Patients table.

In [14]:
query4 = '''
SELECT
    *
FROM Patients
WHERE Gender = 'Female';
'''

show_query(query4)

SQL query:
SELECT
    *
FROM Patients
WHERE Gender = 'Female';


,PatientID,Name,DOB,Gender,Address
0,P002,Maria Lopez,1990-11-03,Female,"892 North Ridge Avenue, Durham, NC 27701"
1,P004,Aisha Khan,1990-09-09,Female,"55 Harbor Point Road, Charlotte, NC 28202"


,PatientID,Name,DOB,Gender,Address
0,P002,Maria Lopez,1990-11-03,Female,"892 North Ridge Avenue, Durham, NC 27701"
1,P004,Aisha Khan,1990-09-09,Female,"55 Harbor Point Road, Charlotte, NC 28202"


<details>
    <summary>Click here for solution</summary>

SELECT *
FROM Patients
WHERE Gender = 'Female';

</details>

## Exercise 2
Write an SQL query to list all unique payers from the Visits table.

In [18]:
query5 = '''
SELECT
    Distinct Payer
FROM Visits
ORDER BY
payer;
'''

show_query(query5)

SQL query:
SELECT
    Distinct Payer
FROM Visits
ORDER BY
payer;


,Payer
0,Aetna
1,BlueCross
2,Cigna
3,Medicare


,Payer
0,Aetna
1,BlueCross
2,Cigna
3,Medicare


<details>
    <summary>Click here for solution</summary>

SELECT DISTINCT Payer
FROM Visits
ORDER BY Payer;

</details>

## Exercise 3
Write an SQL query to display patient names along with their visit dates and procedure descriptions. (Hint: Use 'INNER JOIN')

In [19]:
query6 = '''
SELECT p.PatientID, p.Name AS PatientName, p.Gender, v.VisitID, v.VisitDate, v.ProcedureDescription, v.Charge FROM Patients p INNER JOIN Visits v ON p.PatientID = v.PatientID ORDER BY v.VisitDate DESC;
'''

show_query(query6)

SQL query:
SELECT p.PatientID, p.Name AS PatientName, p.Gender, v.VisitID, v.VisitDate, v.ProcedureDescription, v.Charge FROM Patients p INNER JOIN Visits v ON p.PatientID = v.PatientID ORDER BY v.VisitDate DESC;


,PatientID,PatientName,Gender,VisitID,VisitDate,ProcedureDescription,Charge
0,P004,Aisha Khan,Female,V005,2024-02-05,Rapid Strep Test,30.0
1,P001,John Smith,Male,V004,2024-02-01,Comprehensive Metabolic Panel (CMP),55.0
2,P003,Tim Chen,Male,V003,2024-01-20,Venipuncture,15.0
3,P002,Maria Lopez,Female,V002,2024-01-15,Office Visit – Established Patient,90.0
4,P001,John Smith,Male,V001,2024-01-12,Complete Blood Count (CBC),45.0


,PatientID,PatientName,Gender,VisitID,VisitDate,ProcedureDescription,Charge
0,P004,Aisha Khan,Female,V005,2024-02-05,Rapid Strep Test,30.0
1,P001,John Smith,Male,V004,2024-02-01,Comprehensive Metabolic Panel (CMP),55.0
2,P003,Tim Chen,Male,V003,2024-01-20,Venipuncture,15.0
3,P002,Maria Lopez,Female,V002,2024-01-15,Office Visit – Established Patient,90.0
4,P001,John Smith,Male,V001,2024-01-12,Complete Blood Count (CBC),45.0


<details>
    <summary>Click here for solution</summary>

SELECT
    p.PatientID,
    p.Name AS PatientName,
    p.Gender,
    v.VisitID,
    v.VisitDate,
    v.ProcedureDescription,
    v.Charge
FROM Patients p
INNER JOIN Visits v ON p.PatientID = v.PatientID
ORDER BY v.VisitDate DESC;

</details>

## Summary

In this lab, you:

- Designed a relational healthcare database with normalized tables for patients, visits (billing), and lab results  
- Applied primary and foreign keys to link related records and maintain referential integrity  
- Used SQL to retrieve and combine information across tables with `SELECT`, `WHERE`, `ORDER BY`, and `JOIN`  

These patterns reflect how relational databases support efficient, accurate healthcare data management in real-world clinical and operational systems.